# 2-bob. 2-BOB. KIBERHUJUMLAR VA XAVFSIZLIK MA'LUMOTLARI
## Amaliy misol: veb-server jurnalidan xususiyatlar ajratish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/02-bob/2-1_veb-server-jurnalidan-xususiyatlar-ajratish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 2-bob.*

Quyidagi dastur veb-server jurnalidan har bir IP manzil uchun to'rtta xususiyatni hisoblaydi: so'rovlar soni, xato bilan tugagan so'rovlar ulushi, SQL in'yeksiyaga xos belgilar soni va kirish sahifasiga yuborilgan so'rovlar soni. Bunday xususiyatlar keyinchalik mashinali o'qitish modeliga kirish ma'lumoti sifatida beriladi. Dastur amaliy mashg'ulotning A2 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import re
from collections import defaultdict

# Veb-server jurnalidan (access.log) olingan qatorlar
jurnal = """\
192.168.1.10 - - [12/Nov/2026:10:01:02 +0500] "GET /index.html HTTP/1.1" 200 5120
192.168.1.10 - - [12/Nov/2026:10:01:05 +0500] "GET /talabalar.html HTTP/1.1" 200 3300
10.0.0.66 - - [12/Nov/2026:10:02:11 +0500] "GET /baho.php?id=1' OR '1'='1 HTTP/1.1" 500 120
10.0.0.66 - - [12/Nov/2026:10:02:12 +0500] "GET /baho.php?id=1 UNION SELECT login,parol FROM users HTTP/1.1" 500 120
10.0.0.66 - - [12/Nov/2026:10:02:13 +0500] "GET /admin HTTP/1.1" 403 90
172.16.5.4 - - [12/Nov/2026:10:03:00 +0500] "POST /login HTTP/1.1" 401 80
172.16.5.4 - - [12/Nov/2026:10:03:01 +0500] "POST /login HTTP/1.1" 401 80
172.16.5.4 - - [12/Nov/2026:10:03:02 +0500] "POST /login HTTP/1.1" 401 80
172.16.5.4 - - [12/Nov/2026:10:03:03 +0500] "POST /login HTTP/1.1" 200 450
"""

In [2]:
# 1. Har bir qatorni qismlarga ajratuvchi andoza
andoza = re.compile(r'(\S+) - - \[(.*?)\] "(\S+) (.*?) HTTP/\S+" (\d{3}) (\d+)')
sqli_belgilari = re.compile(r"('|union|select|or\s+'1'='1)", re.IGNORECASE)

In [3]:
# 2. Har bir IP manzil uchun xususiyatlarni hisoblaymiz
xususiyat = defaultdict(lambda: {"sorov": 0, "xatolar": 0, "sqli": 0, "login": 0})
for qator in jurnal.splitlines():
    m = andoza.match(qator)
    if not m:
        continue
    ip, vaqt, usul, manzil, holat, hajm = m.groups()
    x = xususiyat[ip]
    x["sorov"] += 1
    if holat.startswith(("4", "5")):
        x["xatolar"] += 1
    if sqli_belgilari.search(manzil):
        x["sqli"] += 1
    if usul == "POST" and manzil == "/login":
        x["login"] += 1

In [4]:
# 3. Natijani jadval ko'rinishida chiqaramiz
print(f"{'IP manzil':<14}{'so‘rov':>8}{'xato %':>8}{'SQLi':>6}{'login':>7}")
for ip, x in xususiyat.items():
    ulush = 100 * x["xatolar"] / x["sorov"]
    print(f"{ip:<14}{x['sorov']:>8}{ulush:>7.0f}%{x['sqli']:>6}{x['login']:>7}")

IP manzil       so‘rov  xato %  SQLi  login
192.168.1.10         2      0%     0      0
10.0.0.66            3    100%     2      0
172.16.5.4           4     75%     0      4


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
IP manzil       so‘rov  xato %  SQLi  login
192.168.1.10         2      0%     0      0
10.0.0.66            3    100%     2      0
172.16.5.4           4     75%     0      4
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 2-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.